# Finance Complaint Triage: QLoRA fine-tuning and baselines (Week 2)

This notebook fine-tunes Qwen2.5-7B-Instruct with QLoRA on `sft/cfpb_train.jsonl`, then scores
the model on the 170 test rows with `eval/run.py`. It also runs the **base-model baselines**
(zero-shot and few-shot) through the same generation code, so every number in the report is
comparable.

Course plan, Track A: Week 1 needs zero-shot **and few-shot** baselines; Week 2 needs at least
two QLoRA runs with different configurations plus one ablation, tracked in W&B or MLflow, and
every team member runs at least one training job. Week 3 compares base + RAG, fine-tuned, and
fine-tuned + RAG, so every evaluation here scores the test set twice: **with** the guidance
passages and **without** them.

| Step | What happens |
| --- | --- |
| 1. Configure | Pick `MODE` (`train` or `baseline`) and one run from `RUNS` |
| 2. Data | Load the fine-tuning files from Drive, apply the run's data option, check token lengths |
| 3. Train | QLoRA with checkpoints on Drive, so a recycled Colab session resumes (`train` only) |
| 4. Evaluate | Greedy answers on test, with and without passages, scored by `eval/run.py` |
| 5. Record | Each run saves its `summary.json`; `runs/runs.csv` is rebuilt from all of them |

**Before you start:** the team shares one Drive folder, `cfpb-capstone`: the owner shares it
with everyone as Editor, and each member adds a shortcut to it in My Drive ("Shared with me" →
right-click → Organize → Add shortcut → My Drive). The owner runs `make_dataset_colab.ipynb`
once with `SAVE_TO_DRIVE = True` and `OUT_DIR = "cfpb-capstone/data"`, so the fine-tuning files
are in `MyDrive/cfpb-capstone/data/sft/` for everyone. Then choose **Runtime → Change runtime
type → T4 GPU** and **Runtime → Run all**. For W&B tracking, add a Colab secret `WANDB_API_KEY`
(key icon on the left); without it the run is tracked with MLflow in `data/runs/mlruns`.

**First time on a GPU:** set `SMOKE_STEPS = 20`. It trains 20 steps, prints the peak GPU memory
and the projected time for the full run, and stops. That is the "short QLoRA test run".

## 1. Configure

In [ ]:
# ---- what to run ----
MODE = "train"            # "train": fine-tune RUN, then evaluate | "baseline": base model only
RUN = "r16_lr2e-4"        # a key of RUNS below (train mode)
FEW_SHOT = 0              # baseline mode: 0 = zero-shot, 3 = three worked examples before each row
SMOKE_STEPS = 0           # > 0: train only this many steps, report memory and time, skip eval
MEMBER = "your-name"      # who ran it: goes into runs.csv

# ---- runs: the plan needs >= 2 configurations and 1 ablation; add rows, keep old ones ----
# data: "mixed" = the files as built (some train rows without passages)
#       "no_oos" = ablation: drop the out-of-scope training rows (does refusal come from them?)
#       "all_passages" = ablation: train only on rows shown passages
RUNS = {
    "r16_lr2e-4":     dict(r=16, alpha=32,  lr=2e-4, epochs=2, modules="all-linear", data="mixed"),
    "r64_lr1e-4":     dict(r=64, alpha=128, lr=1e-4, epochs=2, modules="all-linear", data="mixed"),
    "r16_attn_only":  dict(r=16, alpha=32,  lr=2e-4, epochs=2, modules="attention",  data="mixed"),
    "ablate_no_oos":  dict(r=16, alpha=32,  lr=2e-4, epochs=2, modules="all-linear", data="no_oos"),
}

# ---- fixed for every run, so runs differ only in RUNS ----
BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
MAX_LENGTH = 3072          # longest example is about 2,400 tokens; longer rows are reported, never cut
BATCH, GRAD_ACCUM = 1, 16  # effective batch 16
WARMUP, DROPOUT = 0.03, 0.05
EVAL_EVERY = 25            # steps between validation-loss checks and checkpoints
GEN_BATCH, MAX_NEW_TOKENS = 8, 400
SEED = 42

DATA_DIR = "/content/drive/MyDrive/cfpb-capstone/data"   # shared team folder (see notebooks/README.md)
REPO_URL = "https://github.com/parulrewatkar0692/finance-complaint-triage"
REPO_BRANCH = "main"

In [ ]:
import importlib.util, subprocess, sys
need = ["peft", "trl", "bitsandbytes", "accelerate", "datasets", "wandb", "mlflow"]
missing = [p for p in need if importlib.util.find_spec(p) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", *missing], check=True)

import json, os, random, time, csv
import torch, transformers, peft, trl
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| peft", peft.__version__, "| trl", trl.__version__)
assert torch.cuda.is_available(), "needs a GPU: Runtime -> Change runtime type -> T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

if "google.colab" in sys.modules and DATA_DIR.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
SFT_DIR = os.path.join(DATA_DIR, "sft")
assert os.path.exists(os.path.join(SFT_DIR, "cfpb_train.jsonl")), \
    f"no fine-tuning files in {SFT_DIR}: run make_dataset_colab.ipynb with SAVE_TO_DRIVE = True"

REPO_DIR = os.path.abspath("finance-complaint-triage")
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, REPO_DIR],
                   check=True)

cfg = dict(RUNS[RUN]) if MODE == "train" else dict(few_shot=FEW_SHOT)
NAME = RUN if MODE == "train" else f"base_{FEW_SHOT}shot"
RUN_DIR = os.path.join(DATA_DIR, "runs", NAME)
os.makedirs(RUN_DIR, exist_ok=True)
print("run:", NAME, cfg, "->", RUN_DIR)

## 2. Data

The train and validation files are used as built, except for the run's `data` option. The test
file is scored twice: as built (passages shown) and with the passages removed
(`Guidance documents: none`, no retrieved ids), which is the "no RAG" condition.

In [ ]:
def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


def without_passages(ex):
    """Same example with the guidance documents removed, and no citations in the target."""
    ex = json.loads(json.dumps(ex))
    user = ex["messages"][1]["content"]
    ex["messages"][1]["content"] = user.split("Guidance documents:")[0] + "Guidance documents: none"
    ex["retrieved"], ex["with_passages"] = [], False
    if len(ex["messages"]) > 2:
        t = json.loads(ex["messages"][2]["content"])
        t["citations"] = []
        ex["messages"][2]["content"] = json.dumps(t, ensure_ascii=False)
    return ex


train = read_jsonl(os.path.join(SFT_DIR, "cfpb_train.jsonl"))
val = read_jsonl(os.path.join(SFT_DIR, "cfpb_val.jsonl"))
test = read_jsonl(os.path.join(SFT_DIR, "cfpb_test.jsonl"))
if MODE == "train":
    if cfg["data"] == "no_oos":
        train = [e for e in train if e["source"] != "out_of_scope"]
    elif cfg["data"] == "all_passages":
        train = [e for e in train if e["with_passages"]]
    else:
        assert cfg["data"] == "mixed", cfg["data"]
print(f"train {len(train)} (with passages {sum(e['with_passages'] for e in train)}, "
      f"out of scope {sum(e['source'] == 'out_of_scope' for e in train)}) | val {len(val)} | test {len(test)}")

TEST_SETS = {"rag": os.path.join(SFT_DIR, "cfpb_test.jsonl"),
             "no_rag": os.path.join(RUN_DIR, "cfpb_test_no_passages.jsonl")}
write_jsonl(TEST_SETS["no_rag"], [without_passages(e) for e in test])

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token


def n_tokens(ex):
    return len(tok.apply_chat_template(ex["messages"], tokenize=True))


lengths = sorted(n_tokens(e) for e in train + val)
too_long = [e["id"] for e in train + val if n_tokens(e) > MAX_LENGTH]
print(f"tokens per example: median {lengths[len(lengths) // 2]}, 95th pct "
      f"{lengths[int(0.95 * len(lengths))]}, max {lengths[-1]}")
print(f"over MAX_LENGTH={MAX_LENGTH}: {len(too_long)} (dropped, never truncated: cutting a row "
      "would cut its target)")
train = [e for e in train if e["id"] not in too_long]
val = [e for e in val if e["id"] not in too_long]

## 3. Train (train mode)

Prompt-completion format: the loss is computed on the assistant JSON only, not on the system
prompt and complaint. 4-bit NF4 base, fp16 (a T4 has no bf16), gradient checkpointing,
paged 8-bit AdamW. Checkpoints go to Drive every `EVAL_EVERY` steps and the best by validation
loss is kept, so a recycled session continues where it stopped: just run all again.

In [ ]:
import inspect
from datasets import Dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

BNB = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
MODULES = {"all-linear": "all-linear",
           "attention": ["q_proj", "k_proj", "v_proj", "o_proj"]}


def as_prompt_completion(rows):
    return Dataset.from_list([{"prompt": e["messages"][:2], "completion": e["messages"][2:]}
                              for e in rows])


def tracking():
    try:
        from google.colab import userdata
        os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    except Exception:
        pass
    if os.environ.get("WANDB_API_KEY"):
        os.environ.setdefault("WANDB_PROJECT", "finance-complaint-triage")
        return "wandb"
    os.environ["MLFLOW_TRACKING_URI"] = "file://" + os.path.join(DATA_DIR, "runs", "mlruns")
    os.environ.setdefault("MLFLOW_EXPERIMENT_NAME", "finance-complaint-triage")
    return "mlflow"


ADAPTER_DIR = os.path.join(RUN_DIR, "adapter")
train_stats = {}
if MODE == "train" and not os.path.exists(os.path.join(ADAPTER_DIR, "adapter_config.json")):
    torch.manual_seed(SEED)
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=BNB,
                                                 device_map="auto", torch_dtype=torch.float16)
    tok.padding_side = "right"
    args = dict(
        output_dir=os.path.join(RUN_DIR, "checkpoints"), run_name=f"{NAME}-{MEMBER}",
        num_train_epochs=cfg["epochs"], learning_rate=cfg["lr"],
        per_device_train_batch_size=BATCH, per_device_eval_batch_size=BATCH,
        gradient_accumulation_steps=GRAD_ACCUM, warmup_ratio=WARMUP, lr_scheduler_type="cosine",
        optim="paged_adamw_8bit", fp16=True, bf16=False,
        gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
        eval_strategy="steps", eval_steps=EVAL_EVERY, save_strategy="steps",
        save_steps=EVAL_EVERY, save_total_limit=2, load_best_model_at_end=True,
        metric_for_best_model="eval_loss", logging_steps=5, seed=SEED,
        report_to=tracking() if not SMOKE_STEPS else "none",
        max_steps=SMOKE_STEPS or -1)
    # trl renamed max_seq_length to max_length; pass whichever this version has
    params = inspect.signature(SFTConfig).parameters
    args["max_length" if "max_length" in params else "max_seq_length"] = MAX_LENGTH
    if "completion_only_loss" in params:
        args["completion_only_loss"] = True
    lora = LoraConfig(r=cfg["r"], lora_alpha=cfg["alpha"], lora_dropout=DROPOUT,
                      target_modules=MODULES[cfg["modules"]], task_type="CAUSAL_LM")
    trainer = SFTTrainer(model=model, args=SFTConfig(**args), peft_config=lora,
                         train_dataset=as_prompt_completion(train),
                         eval_dataset=as_prompt_completion(val), processing_class=tok)
    trainer.model.print_trainable_parameters()

    ckpts = [d for d in os.listdir(args["output_dir"]) if d.startswith("checkpoint-")] \
        if os.path.exists(args["output_dir"]) and not SMOKE_STEPS else []
    torch.cuda.reset_peak_memory_stats()
    start = time.time()
    result = trainer.train(resume_from_checkpoint=bool(ckpts) or None)
    minutes = (time.time() - start) / 60
    peak = torch.cuda.max_memory_allocated() / 2**30
    steps = result.global_step
    if SMOKE_STEPS:
        full = cfg["epochs"] * -(-len(train) // (BATCH * GRAD_ACCUM))
        print(f"\nSMOKE: {steps} steps in {minutes:.1f} min, peak GPU memory {peak:.1f} GB")
        print(f"full run: {full} steps, about {minutes / steps * full / 60:.1f} hours "
              "(plus about 15 min per evaluation pass)")
        print("Set SMOKE_STEPS = 0 and run all again for the real run.")
    else:
        trainer.save_model(ADAPTER_DIR)
        train_stats = {"train_minutes": round(minutes, 1), "peak_gb": round(peak, 1),
                       "steps": steps, "best_eval_loss": trainer.state.best_metric}
        json.dump({"run": NAME, "config": cfg, "member": MEMBER, **train_stats,
                   "log_history": trainer.state.log_history},
                  open(os.path.join(RUN_DIR, "train_log.json"), "w"), indent=2)
        print("adapter ->", ADAPTER_DIR, train_stats)
    del trainer, model
    torch.cuda.empty_cache()
elif MODE == "train":
    print("adapter already trained:", ADAPTER_DIR, "(delete it to retrain)")
    if os.path.exists(os.path.join(RUN_DIR, "train_log.json")):
        log = json.load(open(os.path.join(RUN_DIR, "train_log.json")))
        train_stats = {k: log.get(k) for k in ["train_minutes", "peak_gb", "steps", "best_eval_loss"]}

## 4. Evaluate on test, with and without passages

Same generation settings as the Week 1 baseline (4-bit, greedy, left padding, batches sorted by
length, resumable). In baseline mode with `FEW_SHOT = 3`, three fixed training examples (two
real complaints and one out-of-scope message, seed 42) are shown as earlier turns before each
test row. Answers are saved as they come, so a rerun resumes.

In [ ]:
def few_shot_turns(k, rag):
    """k - 1 real examples (from the 100 shortest, to keep prompts within T4 memory) and one
    out-of-scope example, shown with passages only in the rag condition."""
    if not k:
        return []
    rng = random.Random(SEED)
    real = sorted((e for e in train if e["source"] == "real" and e["with_passages"]),
                  key=lambda e: (n_tokens(e), e["id"]))[:100]
    oos = sorted((e for e in train if e["source"] == "out_of_scope"), key=lambda e: e["id"])
    picks = rng.sample(real, k - 1) + rng.sample(oos, 1)
    return [m for e in picks for m in (e if rag else without_passages(e))["messages"][1:]]


def prompts_for(rows, shots):
    return [tok.apply_chat_template([r["messages"][0]] + shots + r["messages"][1:],
                                    tokenize=False, add_generation_prompt=True) for r in rows]


def generate_all(model, rows, pred_path, shots):
    done = set()
    if os.path.exists(pred_path):
        done = {json.loads(l)["id"] for l in open(pred_path, encoding="utf-8") if l.strip()}
    todo = sorted((r for r in rows if r["id"] not in done),
                  key=lambda r: len(r["messages"][1]["content"]))
    print(f"{os.path.basename(pred_path)}: {len(done)} done, {len(todo)} to go")
    start = time.time()
    for i in range(0, len(todo), GEN_BATCH):
        batch = todo[i:i + GEN_BATCH]
        try:
            texts = generate_batch(model, batch, shots)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()          # batch too long for the GPU: one row at a time
            texts = [t for r in batch for t in generate_batch(model, [r], shots)]
        with open(pred_path, "a", encoding="utf-8") as f:
            for r, t in zip(batch, texts):
                f.write(json.dumps({"id": r["id"], "output": t.strip()}, ensure_ascii=False) + "\n")
        print(f"  {i + len(batch)}/{len(todo)}  {(time.time() - start) / 60:.1f} min")


def generate_batch(model, batch, shots):
    enc = tok(prompts_for(batch, shots), return_tensors="pt", padding=True).to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                             temperature=None, top_p=None, top_k=None, pad_token_id=tok.pad_token_id)
    return tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)


if not SMOKE_STEPS:
    from peft import PeftModel
    tok.padding_side = "left"
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=BNB,
                                                 device_map="auto", torch_dtype=torch.float16)
    if MODE == "train":
        model = PeftModel.from_pretrained(model, ADAPTER_DIR)
    model.eval()
    reports = {}
    for cond, path in TEST_SETS.items():
        shots = few_shot_turns(FEW_SHOT if MODE == "baseline" else 0, rag=cond == "rag")
        rows = read_jsonl(path)
        pred = os.path.join(RUN_DIR, f"predictions_{cond}.jsonl")
        start = time.time()
        generate_all(model, rows, pred, shots)
        out = os.path.join(RUN_DIR, f"report_{cond}.json")
        run = subprocess.run([sys.executable, os.path.join(REPO_DIR, "eval", "run.py"), pred,
                              "--test", path, "--out", out], capture_output=True, text=True)
        print(f"\n=== {NAME} | {cond} ===\n", run.stdout, run.stderr)
        run.check_returncode()
        reports[cond] = json.load(open(out))

## 5. Record the run

Each run writes its own `summary.json` (one row per test condition), so members who finish at the
same time on a shared Drive folder never overwrite each other. `runs/runs.csv` is then rebuilt
from every run's summary: the comparison table for the report (baselines, the configurations and
the ablation). Rerun just this cell any time to refresh it.

In [ ]:
import glob
import pandas as pd

KEYS = ["json_valid", "product_acc", "product_macro_f1", "issue_acc", "oos_recall",
        "false_refusal", "citation_valid", "reply_rules"]
if not SMOKE_STEPS:
    summary = []
    for cond, rep in reports.items():
        m = rep["all"]["metrics"]
        summary.append({"run": NAME, "member": MEMBER, "test": cond, "config": json.dumps(cfg),
                        **{k: None if m[k]["value"] is None else round(m[k]["value"], 3)
                           for k in KEYS},
                        **{k + "_ci95": m[k]["ci95"] and [round(x, 3) for x in m[k]["ci95"]]
                           for k in KEYS},
                        **{k: train_stats.get(k) for k in ["train_minutes", "peak_gb",
                                                           "best_eval_loss"]},
                        "date": time.strftime("%Y-%m-%d")})
    json.dump(summary, open(os.path.join(RUN_DIR, "summary.json"), "w"), indent=2)

rows = [r for p in sorted(glob.glob(os.path.join(DATA_DIR, "runs", "*", "summary.json")))
        for r in json.load(open(p))]
table = pd.DataFrame(rows)
if len(table):
    path = os.path.join(DATA_DIR, "runs", "runs.csv")
    table.to_csv(path, index=False)
    print(f"{len(table)} rows from {table['run'].nunique()} runs ->", path)
    display(table[["run", "member", "test", *KEYS]])